# Retail Business Performance & Profitability Analysis

**Elevate Labs Data Analyst Internship — Final Project**

Analyzing transactional retail data to uncover profit-draining categories, examine fulfillment-speed vs profitability, and identify seasonal patterns.

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

sns.set_style('whitegrid')

df = pd.read_csv('superstore.csv', encoding='latin1')
df.head()

## 2. Data Cleaning
Check for missing/null values and fix them.

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing Product Base Margin with category-level median
df['Product Base Margin'] = df.groupby('Product Category')['Product Base Margin'].transform(
    lambda x: x.fillna(x.median())
)

# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Remove invalid rows (non-positive sales)
df = df[df['Sales'] > 0]
print("Cleaned shape:", df.shape)

## 3. Feature Engineering

In [ ]:
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Days to Ship'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Profit Margin %'] = (df['Profit'] / df['Sales']) * 100

def get_season(month):
    if month in [12,1,2]: return 'Winter'
    elif month in [3,4,5]: return 'Spring'
    elif month in [6,7,8]: return 'Summer'
    else: return 'Fall'

df['Season'] = df['Order Month'].apply(get_season)
df.to_csv('superstore_cleaned.csv', index=False)
df.head()

## 4. SQL Analysis (using SQLite in-memory — same queries as SSMS .sql file)

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')

profit_by_category = pd.read_sql_query('''
SELECT [Product Category] AS Category,
       ROUND(SUM(Sales),2) AS Total_Sales,
       ROUND(SUM(Profit),2) AS Total_Profit,
       ROUND(SUM(Profit)*100.0/SUM(Sales),2) AS Profit_Margin_Pct
FROM sales
GROUP BY [Product Category]
ORDER BY Profit_Margin_Pct DESC;
''', conn)
profit_by_category

In [ ]:
profit_by_subcategory = pd.read_sql_query('''
SELECT [Product Category] AS Category,
       [Product Sub-Category] AS Sub_Category,
       ROUND(SUM(Sales),2) AS Total_Sales,
       ROUND(SUM(Profit),2) AS Total_Profit,
       ROUND(SUM(Profit)*100.0/SUM(Sales),2) AS Profit_Margin_Pct
FROM sales
GROUP BY [Product Category], [Product Sub-Category]
ORDER BY Profit_Margin_Pct ASC
LIMIT 10;
''', conn)
profit_by_subcategory

In [ ]:
profit_by_region = pd.read_sql_query('''
SELECT Region,
       ROUND(SUM(Sales),2) AS Total_Sales,
       ROUND(SUM(Profit),2) AS Total_Profit,
       ROUND(SUM(Profit)*100.0/SUM(Sales),2) AS Profit_Margin_Pct
FROM sales
GROUP BY Region
ORDER BY Profit_Margin_Pct DESC;
''', conn)
profit_by_region

## 5. Correlation: Fulfillment Speed (Days to Ship) vs Profitability
Using 'Days to Ship' as a proxy for inventory/fulfillment efficiency.

In [ ]:
turnover = df.groupby('Product Sub-Category').agg(
    Avg_Days_To_Ship=('Days to Ship','mean'),
    Avg_Profit_Margin=('Profit Margin %','mean'),
    Total_Sales=('Sales','sum')
).reset_index()

corr = turnover['Avg_Days_To_Ship'].corr(turnover['Avg_Profit_Margin'])
print(f"Correlation (sub-category level): {corr:.3f}")

corr_row = df['Days to Ship'].corr(df['Profit Margin %'])
print(f"Correlation (order level): {corr_row:.3f}")

turnover.sort_values('Avg_Days_To_Ship', ascending=False)

## 6. Visualizations

In [ ]:
plt.figure(figsize=(7,4.5))
sns.barplot(data=profit_by_category, x='Category', y='Profit_Margin_Pct', hue='Category', palette='viridis', legend=False)
plt.title('Profit Margin % by Product Category')
plt.ylabel('Profit Margin (%)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,4.5))
sns.barplot(data=profit_by_region, x='Profit_Margin_Pct', y='Region', hue='Region', palette='mako', legend=False)
plt.title('Profit Margin % by Region')
plt.xlabel('Profit Margin (%)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,4.5))
sns.scatterplot(data=turnover, x='Avg_Days_To_Ship', y='Avg_Profit_Margin', size='Total_Sales', legend=False, sizes=(50,400))
plt.title('Avg Days to Ship vs Avg Profit Margin (by Sub-Category)')
plt.xlabel('Avg Days to Ship')
plt.ylabel('Avg Profit Margin (%)')
plt.tight_layout()
plt.show()

## 7. Key Insights & Strategic Recommendations

1. **Technology (14.8%) and Office Supplies (13.8%)** are the most profitable categories, while **Furniture is barely profitable at 2.3%** — dragged down heavily by Tables and Bookcases, which post consistent losses.
2. **Tables (-$191K) and Bookcases (-$99K)** are the top loss-making sub-categories — likely over-discounted or overstocked. Same for Office Machines (-$204K).
3. **Nunavut and Yukon regions have the weakest profit margins** (2.4% and 7.6%), while Northwest Territories and Atlantic perform best (>11%).
4. **Fulfillment speed (Days to Ship) shows only a weak correlation (~0.29) with profit margin at the category level**, and almost none at the individual order level — meaning shipping speed alone isn't driving losses; pricing/discounting strategy is the bigger lever.
5. **Recommendation:** Re-evaluate discount policy on Furniture (especially Tables/Bookcases), audit Nunavut/Yukon regional pricing, and consider phasing out or repricing the worst-performing sub-categories.